In [1]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path

import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

DATASET_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "uah_dataset_window200.npz"
)

sensor_data = np.load(DATASET_PATH)

X = sensor_data["X"]
y = sensor_data["y"]
groups = sensor_data["groups"]

print(X.shape)

(30356, 200, 13)


In [5]:
unique_groups = np.unique(groups)

trip_labels = np.array([
    y[groups == trip][0]
    for trip in unique_groups
])

train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
    stratify=trip_labels,
)

In [6]:
train_mask = np.isin(
    groups,
    train_groups,
)

test_mask = np.isin(
    groups,
    test_groups,
)

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

In [7]:
print(X_train.shape)
print(X_test.shape)

(24469, 200, 13)
(5887, 200, 13)


In [8]:
from sklearn.preprocessing import StandardScaler

# (samples, timesteps, features)
n_train, seq_len, n_features = X_train.shape
n_test = X_test.shape[0]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(n_train, seq_len, n_features)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(n_test, seq_len, n_features)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(24469, 200, 13)
(5887, 200, 13)


In [9]:
from torch.utils.data import Dataset

class DriverDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [10]:
from torch.utils.data import DataLoader

batch_size = 32

train_dataset = DriverDataset(X_train_scaled, y_train)
test_dataset = DriverDataset(X_test_scaled, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [11]:
import torch.nn as nn

class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=13,
        hidden_size=64,
        num_layers=2,
        num_classes=3,
        dropout=0.3
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        x = hidden[-1]

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [13]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print(class_weights)

tensor([0.8212, 0.9861, 1.3017])


In [14]:
model = LSTMClassifier().to(device)

In [15]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [17]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [18]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    loss = running_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return loss, acc, macro_f1, all_labels, all_preds

In [19]:
import copy

num_epochs = 20
patience = 3

best_model = None
best_f1 = 0.0
patience_counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1, _, _ = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    test_f1s.append(test_f1)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f} | "
        f"Macro F1: {test_f1:.4f}"
    )

    if test_f1 > best_f1:

        best_f1 = test_f1
        patience_counter = 0

        best_model = copy.deepcopy(model.state_dict())


    else:

        patience_counter += 1

    if patience_counter >= patience:

        print("\nEarly stopping!")
        break

Epoch 01/20 | Train Loss: 0.9187 | Train Acc: 0.5069 | Test Loss: 0.8585 | Test Acc: 0.5149 | Macro F1: 0.5261
Epoch 02/20 | Train Loss: 0.7174 | Train Acc: 0.6680 | Test Loss: 0.8340 | Test Acc: 0.5689 | Macro F1: 0.5878
Epoch 03/20 | Train Loss: 0.6301 | Train Acc: 0.7216 | Test Loss: 0.7809 | Test Acc: 0.6448 | Macro F1: 0.6607
Epoch 04/20 | Train Loss: 0.5889 | Train Acc: 0.7449 | Test Loss: 0.7131 | Test Acc: 0.6793 | Macro F1: 0.6967
Epoch 05/20 | Train Loss: 0.5531 | Train Acc: 0.7598 | Test Loss: 0.6231 | Test Acc: 0.7226 | Macro F1: 0.7255
Epoch 06/20 | Train Loss: 0.5205 | Train Acc: 0.7727 | Test Loss: 0.6533 | Test Acc: 0.7201 | Macro F1: 0.7256
Epoch 07/20 | Train Loss: 0.4872 | Train Acc: 0.7883 | Test Loss: 0.6536 | Test Acc: 0.6883 | Macro F1: 0.7012
Epoch 08/20 | Train Loss: 0.4657 | Train Acc: 0.7952 | Test Loss: 0.6406 | Test Acc: 0.7353 | Macro F1: 0.7227
Epoch 09/20 | Train Loss: 0.4378 | Train Acc: 0.8100 | Test Loss: 0.7713 | Test Acc: 0.6849 | Macro F1: 0.6999



In [20]:
model.load_state_dict(best_model)

<All keys matched successfully>

In [21]:
test_loss, test_acc, test_f1, y_true, y_pred = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"\nFinal Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")


Final Accuracy : 0.7201
Final Macro F1 : 0.7256


In [22]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)

print(cm)

              precision    recall  f1-score   support

           0       0.76      0.67      0.71      2923
           1       0.55      0.66      0.60      1479
           2       0.85      0.89      0.87      1485

    accuracy                           0.72      5887
   macro avg       0.72      0.74      0.73      5887
weighted avg       0.73      0.72      0.72      5887

[[1944  745  234]
 [ 506  973    0]
 [ 100   63 1322]]


# Experiment 2 - Dropout Analysis

In [23]:
# Dropout Experiment (0.2)

model = LSTMClassifier(
    input_size=X_train.shape[2],
    hidden_size=64,
    num_layers=2,
    num_classes=len(np.unique(y)),
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [24]:
from sklearn.metrics import accuracy_score, f1_score

In [25]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [26]:
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [27]:
num_epochs = 20

best_f1 = 0
best_model = copy.deepcopy(model.state_dict())
patience = 3
counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

train_f1s = []
test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1 = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    train_f1s.append(train_f1)
    test_f1s.append(test_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

    if test_f1 > best_f1:
        best_f1 = test_f1
        best_model = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping!")
        break

Epoch 1/20
Train Loss: 0.9005 | Acc: 0.5323 | F1: 0.5184
Test  Loss: 0.9023 | Acc: 0.5130 | F1: 0.5285
Epoch 2/20
Train Loss: 0.7027 | Acc: 0.6612 | F1: 0.6633
Test  Loss: 0.8758 | Acc: 0.5534 | F1: 0.5808
Epoch 3/20
Train Loss: 0.6307 | Acc: 0.7060 | F1: 0.7135
Test  Loss: 0.8287 | Acc: 0.5633 | F1: 0.5922
Epoch 4/20
Train Loss: 0.5759 | Acc: 0.7306 | F1: 0.7392
Test  Loss: 0.8262 | Acc: 0.5609 | F1: 0.5852
Epoch 5/20
Train Loss: 0.5265 | Acc: 0.7539 | F1: 0.7637
Test  Loss: 0.7003 | Acc: 0.6740 | F1: 0.6914
Epoch 6/20
Train Loss: 0.4878 | Acc: 0.7793 | F1: 0.7886
Test  Loss: 0.7348 | Acc: 0.7055 | F1: 0.7146
Epoch 7/20
Train Loss: 0.4530 | Acc: 0.8070 | F1: 0.8152
Test  Loss: 0.7772 | Acc: 0.7007 | F1: 0.7145
Epoch 8/20
Train Loss: 0.4231 | Acc: 0.8284 | F1: 0.8356
Test  Loss: 0.7999 | Acc: 0.7005 | F1: 0.7106
Epoch 9/20
Train Loss: 0.4014 | Acc: 0.8335 | F1: 0.8417
Test  Loss: 0.9352 | Acc: 0.6056 | F1: 0.6310
Early stopping!


In [28]:
model.load_state_dict(best_model)

test_loss, test_acc, test_f1 = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("=" * 50)
print("Dropout = 0.2")
print(f"Final Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")
print("=" * 50)

Dropout = 0.2
Final Accuracy : 0.7055
Final Macro F1 : 0.7146


In [29]:
# Dropout Experiment (0.4)

model = LSTMClassifier(
    input_size=X_train.shape[2],
    hidden_size=64,
    num_layers=2,
    num_classes=len(np.unique(y)),
    dropout=0.4
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [30]:
num_epochs = 20

best_f1 = 0
best_model = copy.deepcopy(model.state_dict())
patience = 3
counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

train_f1s = []
test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1 = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    train_f1s.append(train_f1)
    test_f1s.append(test_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

    if test_f1 > best_f1:
        best_f1 = test_f1
        best_model = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping!")
        break

Epoch 1/20
Train Loss: 0.9244 | Acc: 0.5108 | F1: 0.5148
Test  Loss: 0.8519 | Acc: 0.5087 | F1: 0.5216
Epoch 2/20
Train Loss: 0.6998 | Acc: 0.6696 | F1: 0.6773
Test  Loss: 0.8872 | Acc: 0.5820 | F1: 0.5986
Epoch 3/20
Train Loss: 0.6181 | Acc: 0.7478 | F1: 0.7546
Test  Loss: 0.8410 | Acc: 0.6249 | F1: 0.6407
Epoch 4/20
Train Loss: 0.5865 | Acc: 0.7647 | F1: 0.7716
Test  Loss: 0.7284 | Acc: 0.6987 | F1: 0.7056
Epoch 5/20
Train Loss: 0.5495 | Acc: 0.7768 | F1: 0.7847
Test  Loss: 0.7018 | Acc: 0.7304 | F1: 0.7271
Epoch 6/20
Train Loss: 0.5223 | Acc: 0.7909 | F1: 0.7983
Test  Loss: 0.6808 | Acc: 0.7258 | F1: 0.7393
Epoch 7/20
Train Loss: 0.4972 | Acc: 0.7995 | F1: 0.8079
Test  Loss: 0.6968 | Acc: 0.6801 | F1: 0.7017
Epoch 8/20
Train Loss: 0.4828 | Acc: 0.7918 | F1: 0.8014
Test  Loss: 0.7572 | Acc: 0.6880 | F1: 0.7051
Epoch 9/20
Train Loss: 0.4537 | Acc: 0.8088 | F1: 0.8176
Test  Loss: 0.7183 | Acc: 0.6835 | F1: 0.7012
Early stopping!


In [31]:
model.load_state_dict(best_model)

test_loss, test_acc, test_f1 = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("=" * 50)
print("Dropout = 0.4")
print(f"Final Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")
print("=" * 50)

Dropout = 0.4
Final Accuracy : 0.7258
Final Macro F1 : 0.7393
